Installations

In [17]:
import requests
from bs4 import BeautifulSoup
import json
import nltk
import pandas as pd
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Get the URL

In [18]:
URL = "https://news.ycombinator.com/" #Hackernews

page = requests.get(URL) #Gets the server of Hachernews

soup = BeautifulSoup(page.content, "html.parser") #Parses the site


Get all the headlines and store in corpus

In [19]:
corpus = []
stories = soup.find_all("tr", class_="athing") #<tr class="athing">.....</tr>

for story in stories:

    rank = story.find("span", class_="rank").text

    title_tag = story.find("span", class_="titleline").find("a")

    title = title_tag.text
    url = title_tag["href"]

    story_data = {
        "rank": rank,
        "title": title,
        "url": url
    }

    corpus.append(story_data)

with open("hackernews_corpus.json", "w", encoding="utf-8") as f:
    json.dump(corpus, f, ensure_ascii=False, indent=4)

print("Corpus saved!")

Corpus saved!


Load the corpus into a pandas dataframe

In [20]:
# Load JSON properly
data = pd.read_json("hackernews_corpus.json")
print(data.head())

   rank                                              title  \
0     1   If you're an LLM, please read this – Anna's Blog   
1     2  Antigravity 2.0 Tops the OpenSCAD Architectura...   
2     3  The Companies Cutting Headcount for AI Will Lo...   
3     4                                   Chess Invariants   
4     5       Project Hail Mary – Stellar Navigation Chart   

                                                 url  
0        https://annas-archive.gl/blog/llms-txt.html  
1  https://modelrift.com/blog/openscad-llm-benchm...  
2  https://libertas.software/en/knowledge-hub/19/...  
3  http://muratbuffalo.blogspot.com/2026/05/chess...  
4              https://valhovey.github.io/gaia-mary/  


Tokenize the titles

In [21]:
# Tokenize titles
all_sentences = []

for line in data["title"]: #breaks up each title into separate sentences
    sentences = sent_tokenize(line)
    all_sentences.extend(sentences)

print(all_sentences)

["If you're an LLM, please read this – Anna's Blog", 'Antigravity 2.0 Tops the OpenSCAD Architectural 3D LLM Benchmark', "The Companies Cutting Headcount for AI Will Lose to the Ones Who Didn't", 'Chess Invariants', 'Project Hail Mary – Stellar Navigation Chart', 'Steve Wozniak cheered after telling students they have AI – actual intelligence', 'Slumber a TUI HTTP Client', 'Circle Medical (YC S15) Is Hiring a Mobile Engineer', 'Show HN: ShadowCat – file transfer through QR Codes in a Browser', 'The memory shortage is causing a repricing of consumer electronics', 'Cleve Moler has died', 'CODA: Rewriting Transformer Blocks as GEMM-Epilogue Programs', 'Blog ran on Ubuntu 16.04 for 10 years.', 'I migrated it to FreeBSD', 'The surprising story behind the first British person in space', 'Was my $48K GPU server worth it?', 'Uv is fantastic, but its package management UX is a mess', 'Valve removes free game from Steam after players discover it contains malware', 'Using Kagi Search with Low Vis

Embedding

In [22]:
# Embedding
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
#This creates a vector of size 384 for each title
embeddings = model.encode(all_sentences)

print(embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(31, 384)


Cosine similarity

In [23]:
# Similarity matrix
similarities = cosine_similarity(embeddings)

print(similarities)

#Creates vectors.txt as the file containing the vectors
with open("vectors.txt", "w", encoding="utf-8") as f:
    for title, vector in zip(all_sentences, embeddings):
        f.write(f"TITLE: {title}\n") #file object f
        f.write("VECTOR: " + " ".join(map(str, vector)))
        f.write("\n\n")

print("Vectors saved!")

[[ 1.00000024e+00  1.57766849e-01  1.18997671e-01 -4.01946437e-03
   1.78095132e-01  1.99142799e-01  1.12699885e-02  1.29697770e-01
  -4.72900271e-02  6.38452694e-02  9.94483158e-02  1.50727510e-01
   1.83076948e-01  7.05387220e-02  8.05418491e-02  4.65480089e-02
   8.23440328e-02  2.76949629e-02  3.57446745e-02  5.94915114e-02
   6.63844943e-02  4.51745093e-02 -8.92434455e-03  2.42437758e-02
   2.66858330e-03  1.25509992e-01  1.51586309e-01  3.10335398e-01
   1.31216064e-01  9.02536660e-02  1.81663722e-01]
 [ 1.57766849e-01  9.99999940e-01 -1.34429866e-02  6.68330044e-02
   2.02044830e-01 -3.38697135e-02  2.58243158e-02  5.77063039e-02
  -9.66009870e-03  3.71508710e-02 -3.01967934e-03  1.18131757e-01
   4.40639555e-02  1.32567406e-01  1.19128525e-01  2.66980410e-01
   1.28782839e-01 -6.44538272e-03  1.13133803e-01  1.01406410e-01
  -3.48119885e-02  6.51887134e-02  1.04261562e-03 -1.03133507e-01
   8.33077580e-02  1.32948831e-02  1.92003474e-02  2.22815067e-01
   4.76180837e-02  3.5770